# Project 1: FDA Recall Announcements

This project collects recall announcements from the FDA Recalls, Market Withdrawals, and Safety Alerts website.

The automated search selects two product types: Food & Beverages and Drugs. For each product type, the program displays 100 entries per page, extracts the table, and clicks Next until it reaches the final page.

The dataset contains the seven columns displayed in the recall table. Each observation represents a listing entry, which may describe multiple products or brands.

Collection is restricted to these two product types and the records available
in the main table at collection time. Archived announcements and information
inside individual announcement pages are excluded.

In [7]:
# Import packages
import time
import pandas as pd

from datetime import datetime
from selenium import webdriver
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

## Access and collection method

Selenium selects the product type, changes the number of displayed entries,
and clicks the pagination links.

The program allows up to 30 seconds for table updates. This is a maximum
waiting time: it continues when the updated rows are available.

A separate 30-second pause is included before actions that request new data,
following the crawl delay stated in the FDA robots.txt file. The FDA robots.txt specifies a 30-second crawl delay for general crawlers. That pause is separate from Selenium’s loading timeout.

In [8]:
# Set up the browser
url = "https://www.fda.gov/safety/recalls-market-withdrawals-safety-alerts"

driver = webdriver.Chrome()
wait = WebDriverWait(driver, 30)

driver.get(url)

row_xpath = '//table[@id="datatable"]/tbody/tr[td[2]/a]'
wait.until(EC.presence_of_element_located(("xpath", row_xpath)))

<selenium.webdriver.remote.webelement.WebElement (session="6844f3160c1458828e4f1a7339102468", element="f.B86FC0D623B0596A7E6893B739E5D5C3.d.B0DD573DCBE0041A55365B1E269D59FE.e.238")>

In [9]:
# Specify the info being extracted
columns = [
    "Date",
    "Brand Name(s)",
    "Product Description",
    "Product Type",
    "Recall Reason Description",
    "Company Name",
    "Terminated Recall"]

product_types = ["Food & Beverages", "Drugs"]
records = []

## Wait for table updates

After changing a filter or clicking Next, the program waits for the old
table rows to be replaced before extracting the new results.

In [10]:
def wait_for_rows(old_row):
    wait.until(EC.staleness_of(old_row))
    wait.until(EC.presence_of_element_located(("xpath", row_xpath)))

## Display 100 entries

The program checks the page-size setting before extracting each page.
It selects 100 only when the current setting is different.

In [12]:
def show_100_entries():
    menu = Select(driver.find_element("xpath", '//select[@name="datatable_length"]'))

    if menu.first_selected_option.get_attribute("value") != "100":
        old_row = driver.find_element("xpath", row_xpath)
        time.sleep(30)
        menu.select_by_visible_text("100")
        wait_for_rows(old_row)

## Extract the table

For each row, the program extracts the seven listing fields. Text spacing
is cleaned, while blank fields remain blank.

The announcement URL is temporarily retained to help identify duplicate
records. It is excluded from the final CSV.

In [13]:
def extract_page():
    rows = driver.find_elements("xpath", row_xpath)

    for row in rows:
        cells = row.find_elements("xpath", "./td")
        record = {}

        for i, column in enumerate(columns):
            text = cells[i].get_attribute("textContent")
            record[column] = " ".join(text.split())

        record["_url"] = cells[1].find_element("xpath", ".//a").get_attribute("href")
        records.append(record)

## Collect every results page

The program selects a product type and collects its results page by page.
It stops when the Next button is disabled, then repeats the process for
the next product type.

In [ ]:
collection_started = datetime.now()

In [16]:
def collect_product_type(product_type):
    old_row = driver.find_element("xpath", row_xpath)
    menu = Select(driver.find_element(
        "xpath", '//select[@id="edit-field-regulated-product-field"]'
    ))

    time.sleep(30)
    menu.select_by_visible_text(product_type)
    wait_for_rows(old_row)

    page = 1

    while True:
        show_100_entries()
        extract_page()
        print(f"{product_type}: page {page} collected")

        next_button = driver.find_element("xpath", '//*[@id="datatable_next"]')

        if "disabled" in next_button.get_attribute("class"):
            break

        old_row = driver.find_element("xpath", row_xpath)
        time.sleep(30)
        next_button.find_element("xpath", "./a").click()
        wait_for_rows(old_row)
        page += 1

for product_type in product_types:
    collect_product_type(product_type)

collection_finished = datetime.now()

Food & Beverages: page 1 collected
Food & Beverages: page 2 collected
Food & Beverages: page 3 collected
Food & Beverages: page 4 collected
Food & Beverages: page 5 collected
Food & Beverages: page 6 collected
Food & Beverages: page 7 collected
Food & Beverages: page 8 collected
Drugs: page 1 collected
Drugs: page 2 collected


## Create and save the dataset

Results from both searches are combined into one dataset. Exact duplicate
records, including their source URL, are removed before saving.

Blank Terminated Recall values are preserved. A blank value is not
automatically interpreted as an active recall.

In [17]:
recalls = pd.DataFrame(records, columns=columns + ["_url"])

duplicates_removed = recalls.duplicated().sum()
recalls = recalls.drop_duplicates()
recalls = recalls[columns].reset_index(drop=True)

recalls.to_csv("fda_food_drug_recalls.csv", index=False)

print("Collection started:", collection_started)
print("Collection finished:", collection_finished)
print("Duplicate records removed:", duplicates_removed)
print("Observations saved:", len(recalls))

recalls.head()

Collection started: 2026-10-16 21:09:59.856378
Collection finished: 2026-10-16 21:21:44.203195
Duplicate records removed: 2
Observations saved: 965


,Date,Brand Name(s),Product Description,Product Type,Recall Reason Description,Company Name,Terminated Recall
0,10/16/2026,Starfish Seafood,Cold Smoked Salmon,"Food & Beverages, Foodborne Illness",Potential to be contaminated with Listeria mon...,Krasnyi Oktyabr Inc.,
1,10/15/2026,Family Farmstead,Raw Mild Cheddar Cheese,"Food & Beverages, Foodborne Illness",Potential to be contaminated with Shiga toxin-...,"Family Farmstead Dairy, LLC",
2,10/14/2026,Para Bien,Para Bien Al Grano,"Food & Beverages, Allergens",Undeclared sesame,Pan Pepin Inc.,
3,10/14/2026,Para Bien,"Para Bien Al Grano, 24oz box","Food & Beverages, Allergens",Undeclared sesame,Pan Pepin Inc.,
4,10/09/2026,Riad's Oliva to Go,"Baba Ghanouj, Spicy Baba Ghanouj, and Hummus","Food & Beverages, Allergens",Undeclared sesame allergen,Oliva LLC.,
